# 08 — Google Search Grounding & Session Management

This notebook covers two important production concerns:

**Part A — Google Search Grounding**  
The model's knowledge has a training cutoff. For real-time facts — stock prices,
sports results, current events — you need **grounding**: the model performs
live Google Searches and synthesizes the results before answering.

**Part B — Session Management**  
Live API sessions have a 15-minute limit. Long-running applications must
handle the `GoAway` signal gracefully, reconnect, and preserve conversational
context across session boundaries.

---

### Topics
| # | Topic | Key API |
|---|-------|---------|
| A1 | Google Search grounding | `types.Tool(google_search=...)` |
| A2 | Search + function calling combined | Multiple tools in one session |
| B1 | Session lifecycle | `resp.go_away` signal |
| B2 | Reconnect loop | `while reconnect_count < max_reconnects` |
| B3 | Context preservation | Inject conversation log into new session |

---

**Notebook structure**
1. Setup  
2. Part A: Google Search grounding  
   - Demo 1: Basic search grounding  
   - Demo 2: Search + function calling together  
3. Part B: Session management  
   - Session lifecycle explanation  
   - Demo 3: Detecting GoAway and reconnecting  
   - Demo 4: Context preservation across reconnects  
4. Best practices  
5. Key takeaways

## Setup

In [1]:
!pip install -q google-genai numpy

In [2]:
import nest_asyncio; nest_asyncio.apply()

import asyncio
import os
import json
import time
import numpy as np
import IPython.display as ipd
from google import genai
from google.genai import types

from dotenv import load_dotenv
load_dotenv()  # loads GEMINI_API_KEY from .env

API_KEY = os.environ.get("GEMINI_API_KEY", "")
MODEL   = "gemini-3.1-flash-live-preview"

client = genai.Client(api_key=API_KEY)
print(f"Client ready. Model: {MODEL}")

Client ready. Model: gemini-3.1-flash-live-preview


In [3]:
# ── Shared helpers ────────────────────────────────────────────────────────────

def play_pcm(raw_bytes: bytes, rate: int = 24000) -> ipd.Audio:
    """Wrap raw PCM bytes in an IPython Audio widget."""
    arr = np.frombuffer(raw_bytes, dtype=np.int16).astype(np.float32) / 32768.0
    return ipd.Audio(arr, rate=rate, autoplay=False)


async def collect_response(session, label: str = "") -> dict:
    """
    Drain a session receive() loop, handling:
      - Audio data
      - Output transcription (text alongside audio)
      - Tool calls (function calling)
      - turn_complete / go_away signals

    Returns dict with keys:
      'audio'       : bytes
      'transcript'  : str
      'tool_calls'  : list of ToolCall objects
      'go_away'     : bool
    """
    audio_chunks = []
    transcript_parts = []
    tool_calls = []
    got_go_away = False

    async for resp in session.receive():
        if resp.data:
            audio_chunks.append(resp.data)

        # Tool / function calls — break immediately so caller can send tool response.
        # If we kept waiting for turn_complete here, the server would also wait
        # for our tool response → deadlock.
        if resp.tool_call:
            for fc in resp.tool_call.function_calls:
                tool_calls.append(fc)
                print(f"[{label}] Tool call: {fc.name}({fc.args})")
            break  # return to caller so it can send tool responses

        sc = resp.server_content
        if sc:
            # output_transcription carries text from AUDIO modality sessions
            if sc.output_transcription and sc.output_transcription.text:
                transcript_parts.append(sc.output_transcription.text)
            if sc.turn_complete:
                if label:
                    print(f"[{label}] Turn complete.")
                break
            if sc.interrupted:
                print(f"[{label}] Interrupted.")
                break

        if resp.go_away:
            got_go_away = True
            print(f"[{label}] GoAway received — session ending soon.")
            break

    return {
        "audio": b"".join(audio_chunks),
        "transcript": "".join(transcript_parts),
        "tool_calls": tool_calls,
        "go_away": got_go_away,
    }


print("Helpers defined.")


Helpers defined.


---
# Part A — Google Search Grounding

## What Is Grounding?

By default the model answers from its training data, which has a cutoff date.
**Grounding** connects the model to live data sources.

When `google_search` tool is enabled:

```
User question
     │
     ▼
  Gemini Live ──── performs Google Search ────► live search results
     │                                                   │
     ◄───────────────────────────────────────────────────┘
     │  synthesizes grounded answer
     ▼
  Response (with citations)
```

The model decides internally whether to search; you just add the tool to config.

### Key Config
```python
tools = [types.Tool(google_search=types.GoogleSearch())]
```

## Demo 1 — Google Search Grounding

In [8]:
async def demo_google_search():
    """Enable Google Search grounding and ask real-time questions."""

    # Enable the Google Search tool
    tools = [types.Tool(google_search=types.GoogleSearch())]

    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
            )
        ),
        tools=tools,
        system_instruction=(
            "You are a knowledgeable assistant with access to real-time Google Search. "
            "Always answer based on the most current information available. "
            "Keep answers concise — 2-3 sentences."
        ),
    )

    # Questions that benefit from real-time search
    questions = [
        "What is the current population of Singapore as of today?",
        "Who won the latest Formula 1 race in 2026 ?",
    ]

    answers = {}

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        for q in questions:
            print(f"\nQ: {q}")
            await session.send_realtime_input(text=q)
            r = await collect_response(session, label="Search")
            answers[q] = r
            print(f"A: {r['transcript'] or '(no text returned)'}")
            if r["audio"]:
                ipd.display(play_pcm(r["audio"]))

    return answers


search_answers = asyncio.run(demo_google_search())

print("\n" + "=" * 60)
print("GROUNDED ANSWERS SUMMARY")
print("=" * 60)
for q, r in search_answers.items():
    print(f"\nQ: {q}")
    print(f"A: {r['transcript']}")
    if r["audio"]:
        ipd.display(play_pcm(r["audio"]))



Q: What is the current population of Singapore as of today?
[Search] Turn complete.
A: As of today, April 22, 2026, Singapore's population is estimated to be around 5.9 to 6.2 million. Official statistics from the middle of last year showed it stood at 6.11 million. This includes residents and a significant non-resident population.



Q: Who won the latest Formula 1 race in 2026 ?
[Search] Turn complete.
A: Kimi Antonelli won the most recent race, the Japanese Grand Prix, for Mercedes. Before that, he also won the Chinese Grand Prix, which was his first F1 victory. The 2026 season started with George Russell winning in Australia.



GROUNDED ANSWERS SUMMARY

Q: What is the current population of Singapore as of today?
A: As of today, April 22, 2026, Singapore's population is estimated to be around 5.9 to 6.2 million. Official statistics from the middle of last year showed it stood at 6.11 million. This includes residents and a significant non-resident population.



Q: Who won the latest Formula 1 race in 2026 ?
A: Kimi Antonelli won the most recent race, the Japanese Grand Prix, for Mercedes. Before that, he also won the Chinese Grand Prix, which was his first F1 victory. The 2026 season started with George Russell winning in Australia.


In [9]:
# ── Inspect the raw tool_calls for the search round-trips ─────────────────
# The google_search tool is handled internally by the model and the server,
# so tool_calls will be empty here. The grounding happens transparently.
# What you get is the final synthesized answer.

print("Tool calls visible to client:")
for q, r in search_answers.items():
    print(f"  {repr(q[:40])}... → {len(r['tool_calls'])} visible tool calls")
print()
print("Note: google_search is a built-in server-side tool. The model queries")
print("it internally; the client does not see the raw search round-trips.")
print("Function calling (Demo 2) is different — those ARE visible to the client.")

Tool calls visible to client:
  'What is the current population of Singap'... → 0 visible tool calls
  'Who won the latest Formula 1 race in 202'... → 0 visible tool calls

Note: google_search is a built-in server-side tool. The model queries
it internally; the client does not see the raw search round-trips.
Function calling (Demo 2) is different — those ARE visible to the client.


---
## Demo 2 — Combining Google Search + Function Calling

You can have **multiple tools active simultaneously**:
- `google_search` — for grounded real-time facts
- A **custom function** — for your own data sources (weather API, internal DB, etc.)

The model decides which tool to use (or both) based on the user's question.

### Flow
```
"Search for latest AI news and tell me Singapore weather"
          │
          ├── google_search → real-time AI news (server-side)
          │
          └── get_weather(city="Singapore") → client receives tool call
                     │
                     ├── client runs the function
                     └── client sends FunctionResponse back
```

In [ ]:
# ── Define a custom weather tool ───────────────────────────────────────────

WEATHER_TOOL = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="get_weather",
            description="Get the current weather for a city.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "city": types.Schema(
                        type=types.Type.STRING,
                        description="The city name, e.g. 'Singapore'",
                    )
                },
                required=["city"],
            ),
        )
    ]
)

GOOGLE_SEARCH_TOOL = types.Tool(google_search=types.GoogleSearch())


def fake_weather_api(city: str) -> dict:
    """Stub that simulates calling a real weather API."""
    # In production: call OpenWeatherMap, WeatherAPI, etc.
    stub_data = {
        "Singapore": {"temp_c": 31, "condition": "Partly cloudy", "humidity": 78},
        "London":    {"temp_c": 14, "condition": "Overcast",      "humidity": 85},
        "New York":  {"temp_c": 22, "condition": "Sunny",          "humidity": 60},
    }
    return stub_data.get(city, {"temp_c": 20, "condition": "Unknown", "humidity": 70})


async def demo_search_plus_function():
    """Google Search + custom function calling in one session."""

    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
            )
        ),
        tools=[GOOGLE_SEARCH_TOOL, WEATHER_TOOL],  # both tools active
        system_instruction=(
            "You are a helpful assistant with access to Google Search and a weather tool. "
            "Use Google Search for news/facts. Use get_weather for weather. "
            "Answer clearly in 3-4 sentences."
        ),
    )

    user_query = (
        "Search for the latest major news about artificial intelligence, "
        "and also tell me the current weather in Singapore."
    )

    final_transcript = ""

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        print(f"User: {user_query}")
        await session.send_realtime_input(text=user_query)

        # ── Handle the response loop, including function calls ──────────────
        while True:
            r = await collect_response(session, label="SearchFn")

            # If there are pending tool calls, handle them and continue
            if r["tool_calls"]:
                function_responses = []
                for fc in r["tool_calls"]:
                    if fc.name == "get_weather":
                        city = fc.args.get("city", "Singapore")
                        weather_data = fake_weather_api(city)
                        print(f"  [Client] Ran get_weather({city!r}) → {weather_data}")
                        function_responses.append(
                            types.FunctionResponse(
                                id=fc.id,
                                name=fc.name,
                                response=weather_data,
                            )
                        )

                # Send all function responses back to the model
                await session.send_tool_response(
                    function_responses=function_responses
                )
                # Continue the loop to get the model's final reply
                continue

            # No tool calls — this is the final answer
            final_transcript = r["transcript"]
            final_audio = r["audio"]
            break

    return final_transcript, final_audio


final_answer, final_audio = asyncio.run(demo_search_plus_function())
print("\n=== Final grounded + weather answer ===")
print(final_answer)
if final_audio:
    ipd.display(play_pcm(final_audio))


User: Search for the latest major news about artificial intelligence, and also tell me the current weather in Singapore.


---
# Part B — Session Management

## Session Lifecycle

Live API sessions are **long-lived WebSocket connections** with these limits:

```
Session opens
    │
    │  Normal conversation (up to ~15 minutes)
    │
    ├── Model sends GoAway signal (time limit approaching)
    │       resp.go_away  is truthy
    │       resp.go_away.time_left  tells you how long you have
    │
    └── Session closes
```

### What You Must Handle

| Signal | Meaning | Action |
|--------|---------|--------|
| `resp.go_away` | Session ending soon | Wrap up, reconnect |
| `sc.interrupted` | Model interrupted itself | May retry or wait |
| `sc.turn_complete` | Normal end of response | Continue conversation |
| Connection error | Network issue | Exponential backoff + reconnect |

### Why Context Matters
When you reconnect, **the new session has no memory of the old one**.
To maintain conversational continuity, you must inject the previous
conversation history into the new session's system prompt or initial turns.

## Demo 3 — Detecting GoAway and Reconnecting

In this demo we simulate the GoAway scenario using a reconnect loop.
Since GoAway only occurs near the 15-minute limit, we simulate the
reconnect logic without actually waiting 15 minutes.

The loop structure below is what you'd use in a real production application.

In [ ]:
async def demo_go_away_reconnect():
    """
    Production-style reconnect loop.
    Handles GoAway by reopening the session up to max_reconnects times.
    """
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
            )
        ),
        system_instruction="You are a helpful assistant. Answer concisely.",
    )

    reconnect_count = 0
    max_reconnects  = 3
    conversation_log = []  # tracks all turns across sessions

    # Questions to ask across potentially multiple sessions
    pending_questions = [
        "What is the speed of light?",
        "What is the largest planet in our solar system?",
        "How many bones are in the human body?",
    ]

    while reconnect_count < max_reconnects and pending_questions:
        print(f"\n--- Session attempt {reconnect_count + 1} ---")

        try:
            async with client.aio.live.connect(model=MODEL, config=config) as session:
                print(f"Session open. {len(pending_questions)} questions remaining.")

                # Ask pending questions one by one
                answered = []
                for q in list(pending_questions):
                    await session.send_realtime_input(text=q)
                    r = await collect_response(session, label=f"S{reconnect_count}")

                    if r["go_away"]:
                        # Session is ending — break out, reconnect will pick up
                        print(
                            f"  GoAway during Q: {repr(q[:40])}..."
                            f" — will reconnect ({reconnect_count + 1}/{max_reconnects})"
                        )
                        reconnect_count += 1
                        break

                    conversation_log.append({"q": q, "a": r["transcript"]})
                    answered.append(q)
                    print(f"  Q: {q}")
                    print(f"  A: {r['transcript']}")
                    if r["audio"]:
                        ipd.display(play_pcm(r["audio"]))

                # Remove answered questions from pending
                for q in answered:
                    pending_questions.remove(q)

                # If all questions answered normally, we're done
                if not pending_questions:
                    print("\nAll questions answered — no reconnect needed.")
                    break

        except Exception as exc:
            print(f"  Connection error: {exc}")
            reconnect_count += 1
            await asyncio.sleep(2 ** reconnect_count)  # exponential back-off

    print(f"\nDone. Reconnect count: {reconnect_count}")
    return conversation_log


log = asyncio.run(demo_go_away_reconnect())
print("\n=== Conversation log ===")
for entry in log:
    print(f"Q: {entry['q']}")
    print(f"A: {entry['a']}")
    print()


---
## Demo 4 — Context Preservation Across Reconnects

When a session ends and you reconnect, the new session has no memory.
To maintain continuity:

1. Store all Q&A turns in a **conversation log**
2. On reconnect, **inject the last N turns** into the new session's
   system prompt so the model has context

### Pattern
```python
# Serialize last 5 turns
context = "\n".join(f"User: {t['q']}\nAssistant: {t['a']}" for t in log[-5:])

# Inject into new session's system prompt
system_with_context = f"""
You are a helpful assistant.

PREVIOUS CONVERSATION (for context only):
{context}

Continue the conversation naturally.
"""
```

In [ ]:
# ── Conversation log (shared across sessions) ──────────────────────────────
conversation_log: list[dict] = []


def build_context_prompt(log: list[dict], max_turns: int = 5) -> str:
    """
    Build a system prompt that includes the last `max_turns` conversation turns.
    Used to inject context into a new session after reconnect.
    """
    base = (
        "You are a helpful assistant with memory of previous conversations. "
        "Answer questions naturally and concisely."
    )
    if not log:
        return base

    recent_turns = log[-max_turns:]
    history_text = "\n".join(
        f"User: {t['q']}\nAssistant: {t['a']}" for t in recent_turns
    )

    return (
        f"{base}\n\n"
        f"PREVIOUS CONVERSATION HISTORY (last {len(recent_turns)} turns):\n"
        f"{history_text}\n\n"
        f"Continue from where the conversation left off."
    )


async def run_session_with_context(questions: list[str]) -> list[dict]:
    """
    Run one session with conversation context injected.
    Returns updated conversation_log.
    """
    global conversation_log

    # Build context-aware system prompt from previous turns
    system_prompt = build_context_prompt(conversation_log)
    print(f"System prompt length: {len(system_prompt)} chars")

    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
            )
        ),
        system_instruction=system_prompt,
    )

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        for q in questions:
            await session.send_realtime_input(text=q)
            r = await collect_response(session, label="CtxSession")

            conversation_log.append({"q": q, "a": r["transcript"]})
            print(f"  Q: {q}")
            print(f"  A: {r['transcript']}")
            if r["audio"]:
                ipd.display(play_pcm(r["audio"]))

            if r["go_away"]:
                print("  GoAway — ending this session.")
                break

    return conversation_log


print("Context-preservation helpers defined.")


In [ ]:
# ── Session 1: First batch of questions ────────────────────────────────────
print("=== Session 1 ===")

batch1 = [
    "My name is Alex and I'm learning about space.",
    "What is a black hole?",
    "How far is the nearest star?",
]

log_after_session1 = asyncio.run(run_session_with_context(batch1))

print(f"\nLog after session 1: {len(log_after_session1)} turns")

In [ ]:
# ── Session 2: Reconnect — context is injected automatically ───────────────
# This simulates what happens after a GoAway / timeout reconnect.
# The new session knows what was said in session 1 via the system prompt.

print("=== Session 2 (simulated reconnect) ===")
print("The new session will know the user's name and previous topics.\n")

batch2 = [
    # Reference previous session — model should know the user's name and prior topics
    "Can you remind me what you told me about black holes?",
    "Also, what's my name?",  # tests that context (name intro) persisted
    "What other space topics would you recommend I study?",
]

log_after_session2 = asyncio.run(run_session_with_context(batch2))

print(f"\nFull log: {len(log_after_session2)} turns across 2 sessions")
print("\n=== Full conversation log ===")
for i, entry in enumerate(log_after_session2, 1):
    print(f"\nTurn {i}")
    print(f"  Q: {entry['q']}")
    print(f"  A: {entry['a']}")

---
## Best Practices for Session Management

### 1. Always Watch for GoAway
```python
async for resp in session.receive():
    if resp.go_away:
        # Handle gracefully — save state, reconnect
        break
```

### 2. Exponential Backoff on Errors
```python
for attempt in range(max_retries):
    try:
        async with client.aio.live.connect(...) as session:
            ...
    except Exception:
        wait = 2 ** attempt   # 1s, 2s, 4s, 8s ...
        await asyncio.sleep(wait)
```

### 3. Keep a Conversation Log
- Store every Q&A turn in a persistent log
- On reconnect, inject the last 5-10 turns as context
- Use the system prompt for context injection (simplest approach)

### 4. Connection Pooling (for High-Traffic Apps)
```python
# Pre-warm a pool of sessions
SESSION_POOL = asyncio.Queue()

async def get_session():
    if SESSION_POOL.empty():
        return await client.aio.live.connect(model=MODEL, config=config).__aenter__()
    return await SESSION_POOL.get()
```

### 5. Health Check Pattern
```python
async def health_check(session) -> bool:
    try:
        await session.send_realtime_input(text="ping")
        r = await asyncio.wait_for(collect_response(session), timeout=3.0)
        return bool(r["transcript"] or r["audio"])
    except asyncio.TimeoutError:
        return False
```

### 6. Context Window Budget
- Each injected turn uses context window tokens
- Limit injected history to last 5-10 turns
- Summarize old turns with another model call if history grows large

In [ ]:
# ── Production-ready reconnect wrapper ─────────────────────────────────────
# This is the full pattern you'd use in a real application.

class LiveSessionManager:
    """
    Manages a Gemini Live session with automatic reconnect and context
    preservation.
    """

    def __init__(self, config: types.LiveConnectConfig, max_reconnects: int = 5):
        self.base_config    = config
        self.max_reconnects = max_reconnects
        self.conv_log: list[dict] = []
        self.reconnect_count = 0

    def _build_config(self) -> types.LiveConnectConfig:
        """Returns config with conversation context injected into system prompt."""
        if not self.conv_log:
            return self.base_config

        recent = self.conv_log[-5:]
        history = "\n".join(f"User: {t['q']}\nAssistant: {t['a']}" for t in recent)

        base_instr = getattr(self.base_config, "system_instruction", "") or ""
        new_instr  = f"{base_instr}\n\nPREVIOUS CONTEXT:\n{history}"

        # Build a new config with the updated system_instruction
        # (LiveConnectConfig is immutable — create a new one)
        return types.LiveConnectConfig(
            response_modalities=["AUDIO"],
            output_audio_transcription=types.AudioTranscriptionConfig(),
            speech_config=types.SpeechConfig(
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
                )
            ),
            tools=self.base_config.tools or [],
            system_instruction=new_instr,
        )

    async def ask(self, question: str) -> str:
        """Send a question, reconnecting if necessary. Returns transcript."""
        for attempt in range(self.max_reconnects):
            try:
                config = self._build_config()
                async with client.aio.live.connect(model=MODEL, config=config) as session:
                    await session.send_realtime_input(text=question)
                    r = await collect_response(session, label=f"Mgr-{attempt}")

                    self.conv_log.append({"q": question, "a": r["transcript"]})

                    if r["go_away"]:
                        self.reconnect_count += 1
                        print(f"GoAway — reconnect #{self.reconnect_count}")
                        continue

                    return r["transcript"], r["audio"]

            except Exception as exc:
                wait = 2 ** attempt
                print(f"Error: {exc} — retrying in {wait}s")
                await asyncio.sleep(wait)

        return "(max reconnects reached)", b""


# Quick demo of the manager
async def demo_manager():
    base_config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        output_audio_transcription=types.AudioTranscriptionConfig(),
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
            )
        ),
        system_instruction="You are a helpful assistant. Be concise.",
    )

    mgr = LiveSessionManager(config=base_config, max_reconnects=3)

    questions = [
        "I'm testing a session manager. Say 'Session 1 works!' to confirm.",
        "What did you say in your previous response?",  # tests context
    ]

    for q in questions:
        print(f"\nQ: {q}")
        answer, audio = await mgr.ask(q)
        print(f"A: {answer}")
        if audio:
            ipd.display(play_pcm(audio))

    print(f"\nTotal reconnects: {mgr.reconnect_count}")
    print(f"Conv log entries: {len(mgr.conv_log)}")


asyncio.run(demo_manager())


---
## Key Takeaways

### Part A — Google Search Grounding

| Concept | Key Point |
|---------|----------|
| **Grounding** | Adds `types.Tool(google_search=types.GoogleSearch())` to config |
| **Transparent** | Server-side tool; client sees the synthesized answer, not raw search results |
| **Multiple tools** | Google Search + function calling can coexist; model picks the right one |
| **Function responses** | Use `send_tool_response(function_responses=[...])` — never `send_client_content` |

### Part B — Session Management

| Concept | Key Point |
|---------|----------|
| **Session limit** | ~15 minutes per session |
| **GoAway signal** | `resp.go_away` — save state and reconnect |
| **Reconnect loop** | `while reconnect_count < max_reconnects: async with client.aio.live.connect(...)` |
| **Context preservation** | Inject last N turns into new session's system prompt |
| **Exponential backoff** | `await asyncio.sleep(2 ** attempt)` on errors |
| **No built-in memory** | New session = blank slate; YOU must preserve and inject context |

### Series Summary

| Notebook | Topic |
|----------|-------|
| 06 | Proactive audio, VAD modes, silence sensitivity |
| 07 | Affective dialog, tone/style, voice personas |
| 08 | Google Search grounding, session management |

### Reference Links
- [Gemini Live API docs](https://ai.google.dev/gemini-api/docs/live)
- [google-genai Python SDK](https://github.com/googleapis/python-genai)
- [Google Search grounding](https://ai.google.dev/gemini-api/docs/grounding)
- [Function calling guide](https://ai.google.dev/gemini-api/docs/function-calling)